# Which score matches which job

There are two jobs in this project, and they fail in different places.

**A carbon report** multiplies intensity by energy. What matters is the level. Bias is the grams per kWh the report is high or low. MAE is the typical size of that miss. RMSE is the same idea with extra weight on the worst hours. A training loss of squared error pushes the fit toward RMSE. Absolute error pushes it toward MAE. Pinball loss, used by a quantile model, charges more for guessing too clean than for guessing too dirty, which is the direction a report should fear if understating emissions is the mistake.

**A shift** only needs the order of hours inside the day. Regret, from the previous notebook, is the extra g/kWh of running in the forecast's clean window. Rank correlation is the same idea for all 24 hours. A perfect level with the noon dip in the wrong place has a lovely MAE and a bad shift. A level that is 40 g/kWh too clean, with the dip in the right place, has a bad MAE and a fine shift.

This notebook asks which reported score moves with which job. The forecasts are the same fixed ones. The question is what to read later, when tuning starts, so a better score is a better decision rather than a different penalty.

In [1]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12

from src.carbon_data import load_model_frame
from src.modeling import (
    TARGET,
    day_scores,
    error_components,
    fixed_forecasts,
    local_day_labels,
    train_test_split,
)

frame = load_model_frame()
train, test = train_test_split(frame)
assert train.index.max() < test.index.min()
forecasts = fixed_forecasts(train, test)
print(
    f"train through {train.index.max():%Y-%m-%d %H} UTC; "
    f"holdout {test.index.min():%Y-%m-%d %H} to {test.index.max():%Y-%m-%d %H} UTC "
    f"({len(test)} hours)"
)
print(
    f"mean carbon {train[TARGET].mean():.0f} g/kWh in training, "
    f"{test[TARGET].mean():.0f} on the holdout"
)
print("forecasts:", ", ".join(forecasts))


/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in ma

train through 2026-09-20 22 UTC; holdout 2026-09-20 23 to 2026-10-18 23 UTC (673 hours)
mean carbon 103 g/kWh in training, 172 on the holdout
forecasts: hour-of-day mean, yesterday, same hour, day-ahead ridge, day-ahead tree, previous hour


/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in ma

In [2]:
model_rows = []
day_rows = []
for name, predicted in forecasts.items():
    daily = day_scores(test, predicted)
    parts = error_components(test[TARGET], predicted)
    model_rows.append(
        {
            "model": name,
            "bias": parts["bias"],
            "mae": parts["mae"],
            "rmse": parts["rmse"],
            "shape_mae": daily["shape_mae"].mean(),
            "regret_4h": daily["regret_4h"].mean(),
            "rank_corr": daily["rank_corr"].mean(),
        }
    )
    labeled = daily.copy()
    labeled["model"] = name
    day_rows.append(labeled)
comparison = pd.DataFrame(model_rows).set_index("model").loc[list(forecasts)]
by_day = pd.concat(day_rows, ignore_index=True)
by_day["abs_level"] = by_day["level_error"].abs()
by_day["rank_miss"] = 1 - by_day["rank_corr"]
comparison.round(2)


,bias,mae,rmse,shape_mae,regret_4h,rank_corr
model,,,,,,
hour-of-day mean,-69.71,70.62,83.74,24.48,1.90,0.84
"yesterday, same hour",-2.32,27.18,33.99,16.68,2.17,0.84
day-ahead ridge,-6.10,25.06,31.88,16.24,1.99,0.86
day-ahead tree,-20.55,30.79,38.49,17.36,2.45,0.84
previous hour,-2.39,10.16,14.60,9.02,0.78,0.93


## Across days, for the day-ahead ridge

Each number is the correlation, over complete holdout days, between that score and the job. A score that tracks the shift will move with 4-hour regret. A score that tracks the report will move with the size of the level miss.

Rank miss is included because it is built from the order of hours. It should track regret. MAE is the score people usually report. The comparison is whether MAE is actually pointing at the shift.

In [3]:
ridge_days = by_day[by_day["model"] == "day-ahead ridge"]
indicators = ["mae", "rmse", "abs_level", "shape_mae", "rank_miss"]
rows = []
for indicator in indicators:
    rows.append(
        {
            "score": indicator,
            "tracks_shift_regret": ridge_days[indicator].corr(ridge_days["regret_4h"]),
            "tracks_report_level": ridge_days[indicator].corr(ridge_days["abs_level"]),
        }
    )
alignment = pd.DataFrame(rows).sort_values("tracks_shift_regret", ascending=False)
shift_score = alignment.iloc[0]
report_rows = alignment[alignment["score"] != "abs_level"]
report_score = report_rows.sort_values("tracks_report_level", ascending=False).iloc[0]
print(
    f"For the shift, {shift_score['score']} tracks 4-hour regret most closely "
    f"(correlation {shift_score['tracks_shift_regret']:.2f})."
)
print(
    f"For the carbon report, {report_score['score']} tracks the size of the level miss most closely "
    f"(correlation {report_score['tracks_report_level']:.2f}), aside from the level miss itself."
)
print("Model order by MAE:", ", ".join(comparison.sort_values("mae").index))
print("Model order by 4-hour regret:", ", ".join(comparison.sort_values("regret_4h").index))
alignment.round(3)


For the shift, rmse tracks 4-hour regret most closely (correlation 0.35).
For the carbon report, rmse tracks the size of the level miss most closely (correlation 0.94), aside from the level miss itself.
Model order by MAE: previous hour, day-ahead ridge, yesterday, same hour, day-ahead tree, hour-of-day mean
Model order by 4-hour regret: previous hour, hour-of-day mean, day-ahead ridge, yesterday, same hour, day-ahead tree


,score,tracks_shift_regret,tracks_report_level
1,rmse,0.352,0.936
0,mae,0.340,0.932
2,abs_level,0.332,1.000
4,rank_miss,0.220,0.042
3,shape_mae,0.183,0.209
